# 4. Visualizing predictions on NMT recordings

For single recordings of the NMT-events **eval** split, this notebook shows:

1. **Topographic maps** of how many windows per electrode are abnormal, ground truth next to prediction.
2. **EEG traces** with ground-truth events (boxes) and model predictions (shading). One panel shows a 10 s segment
   where every abnormal window was detected; the other shows the segment with the most missed events.

It needs a trained NMT checkpoint, the NMT-events windows (with `channel`, `beg`, `end` keys) and, for the traces,
the original NMT EDF files. Requires `matplotlib` and `mne`.

In [ ]:
from pathlib import Path

# Run from the repository root or from notebooks/.
REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

CHECKPOINT = REPO / "runs" / "nmt" / "neurotransformer" / "best.pth"
NPZ_ROOT = REPO / "data" / "nmt" / "eval"
EDF_DIRS = [Path("/path/to/nmt/edf")]   # searched recursively for <recording_id>.edf
RECORDING_IDS = ["0000137", "0000222", "0000519", "0000681", "0000828", "0000864", "0000875", "0000926", "0001071"]
SEGMENT_SEC = 10

In [ ]:
import torch

from neurotransformer.data import get_spec
from neurotransformer.models import build_model

spec = get_spec("nmt")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = build_model("neurotransformer", spec.n_classes).to(device)
model.load_state_dict(torch.load(CHECKPOINT, map_location=device, weights_only=True))
model.eval();

## Topographic maps

In [ ]:
import matplotlib.pyplot as plt

from neurotransformer.visualization import plot_topomap_comparison, predict_recording

timelines = {}
for rid in RECORDING_IDS:
    timelines[rid] = predict_recording(model, str(NPZ_ROOT), rid, spec.label_map, device)
    plot_topomap_comparison(timelines[rid], rid)
    plt.show()

## EEG traces with ground truth and predictions

In [ ]:
from neurotransformer.visualization import (find_segment_perfect_recall, find_segment_with_misses,
                                            plot_eeg_with_predictions)


def find_edf(recording_id):
    for d in EDF_DIRS:
        hits = sorted(d.rglob(f"{recording_id}.edf"))
        if hits:
            return hits[0]
    return None


for rid, timeline in timelines.items():
    edf = find_edf(rid)
    if edf is None:
        print(f"{rid}: EDF not found, skipping")
        continue
    starts = [find_segment_perfect_recall(timeline, SEGMENT_SEC), find_segment_with_misses(timeline, SEGMENT_SEC)]
    try:
        plot_eeg_with_predictions(str(edf), timeline, starts,
                                  [f"{rid}: all abnormal windows detected", f"{rid}: most missed windows"],
                                  duration=SEGMENT_SEC)
    except (OSError, ValueError) as e:  # unreadable / corrupt EDF
        print(f"{rid}: could not read {edf}: {e}")
        continue
    plt.show()